# 📊 02 — Analyse Exploratoire des Données (EDA)
**Projet DataScientest / Liora — Direction de l'Actuariat Vie**

## Objectifs
1. **Analyse temporelle** — évolution des taux d'urgences 2020-2025
2. **Analyse géographique** — carte choroplèthe des risques par département
3. **Analyse de corrélation** — liens entre météo, pollen, qualité de l'air et urgences
4. **Traitement des valeurs manquantes** — stratégie d'imputation avant modélisation

> Toutes les visualisations sont **interactives** (Plotly) — survolez, zoomez, filtrez !

---
## 0. Imports & chargement

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import urllib.request
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

TABLES_DIR = Path("data/tables")
EDA_DIR    = Path("eda")
EDA_DIR.mkdir(exist_ok=True)

# ── Chargement du dataset final ───────────────────────────────────────────────
df = pd.read_parquet(TABLES_DIR / "df_model.parquet")
df["annee_mois_dt"] = pd.to_datetime(df["annee_mois"] + "-01")
df["mois"] = df["annee_mois_dt"].dt.month
df["annee"] = df["annee_mois_dt"].dt.year

MOIS_LABELS = ["Jan","Fév","Mar","Avr","Mai","Jun",
               "Jul","Aoû","Sep","Oct","Nov","Déc"]

Y_COLS = {
    "Allergie":     "taux_urgences_allergie",
    "Asthme":       "taux_urgences_asthme",
    "Bronchiolite": "taux_urgences_bronchiolite",
}
COLORS = {
    "Allergie":     "#E74C3C",
    "Asthme":       "#2980B9",
    "Bronchiolite": "#27AE60",
}

print(f"✅ df_model chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période  : {df['annee_mois'].min()} → {df['annee_mois'].max()}")
print(f"Depts    : {df['dept'].nunique()}")

✅ df_model chargé : 6,912 lignes × 59 colonnes
Période  : 2020-01 → 2025-12
Depts    : 96


---
## 1. ⏱️ Analyse temporelle
### 1.1 Évolution nationale — 3 pathologies

In [ ]:
# Moyenne nationale par mois
df_nat = (
    df.groupby("annee_mois_dt")[list(Y_COLS.values())]
    .mean()
    .reset_index()
)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Urgences {p}" for p in Y_COLS],
    vertical_spacing=0.08
)

for i, (patho, col) in enumerate(Y_COLS.items(), 1):
    h = COLORS[patho].lstrip('#')
    rgba_fill = f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},0.1)"

    # Courbe principale
    fig.add_trace(
        go.Scatter(
            x=df_nat["annee_mois_dt"],
            y=df_nat[col],
            mode="lines",
            name=patho,
            line=dict(color=COLORS[patho], width=2),
            fill="tozeroy",
            fillcolor=rgba_fill,
            hovertemplate="%{x|%b %Y}<br>Taux : %{y:.1f} /100k<extra></extra>"
        ),
        row=i, col=1
    )

    # Zone Covid
    fig.add_vrect(
        x0="2020-03-01", x1="2022-04-01",
        fillcolor="gray", opacity=0.08,
        annotation_text="Covid" if i == 1 else "",
        annotation_position="top left",
        row=i, col=1
    )

fig.update_layout(
    title_text="Évolution nationale des taux d'urgences (2020-2025)",
    height=700,
    showlegend=True,
    template="plotly_white",
    hovermode="x unified"
)
fig.update_yaxes(title_text="Taux /100k hab.")
fig.update_xaxes(title_text="Date", row=3)
fig.show()
fig.write_html(EDA_DIR / "eda_01_evolution_nationale.html")
print(f"💾 eda/eda_01_evolution_nationale.html")

### 1.2 Saisonnalité — profil moyen par mois

In [3]:
df_sais = df.groupby("mois")[list(Y_COLS.values())].mean().reset_index()
df_sais["mois_label"] = df_sais["mois"].apply(lambda m: MOIS_LABELS[m-1])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"Saisonnalité — {p}" for p in Y_COLS]
)

for i, (patho, col) in enumerate(Y_COLS.items(), 1):
    vals = df_sais[col].values
    idx_max = vals.argmax()
    bar_colors = [
        COLORS[patho] if j == idx_max else "#BDC3C7"
        for j in range(12)
    ]
    fig.add_trace(
        go.Bar(
            x=df_sais["mois_label"],
            y=vals,
            marker_color=bar_colors,
            name=patho,
            hovertemplate="%{x}<br>Taux moyen : %{y:.1f}<extra></extra>"
        ),
        row=1, col=i
    )
    # Annotation pic
    fig.add_annotation(
        x=MOIS_LABELS[idx_max],
        y=vals[idx_max] * 1.1,
        text=f"Pic",
        showarrow=True,
        arrowhead=2,
        row=1, col=i
    )

fig.update_layout(
    title_text="Profil saisonnier moyen par pathologie",
    height=450,
    showlegend=False,
    template="plotly_white"
)
fig.update_yaxes(title_text="Taux moyen /100k hab.", col=1)
fig.show()
fig.write_html(EDA_DIR / "eda_02_saisonnalite.html")
print(f"💾 eda/eda_02_saisonnalite.html")

💾 eda_02_saisonnalite.html


### 1.3 Évolution par département — top 5 et bottom 5

In [4]:
# Pour chaque pathologie, afficher l'évolution des 5 depts les plus touchés
for patho, col in Y_COLS.items():
    if col not in df.columns:
        continue

    top5 = df.groupby("dept")[col].mean().nlargest(5).index.tolist()

    df_top = df[df["dept"].isin(top5)].copy()
    df_top_mois = (
        df_top.groupby(["annee_mois_dt", "dept"])[col]
        .mean().reset_index()
    )

    fig = px.line(
        df_top_mois,
        x="annee_mois_dt",
        y=col,
        color="dept",
        title=f"Top 5 départements — Urgences {patho}",
        labels={col: "Taux /100k hab.", "annee_mois_dt": "Date", "dept": "Dept"},
        template="plotly_white",
        height=400
    )
    fig.update_traces(mode="lines", line_width=2)
    fig.add_vrect(
        x0="2020-03-01", x1="2022-04-01",
        fillcolor="gray", opacity=0.07,
        annotation_text="Covid"
    )
    fig.show()
    fig.write_html(EDA_DIR / f"eda_03_top5_depts_{patho.lower()}.html")
    print(f"💾 eda/eda_03_top5_depts_{patho.lower()}.html")

💾 eda_03_top5_depts_allergie.html


💾 eda_03_top5_depts_asthme.html


💾 eda_03_top5_depts_bronchiolite.html


---
## 2. 🗺️ Analyse géographique
### 2.1 Carte choroplèthe — taux moyen par département

In [ ]:
# Télécharger le GeoJSON des départements français
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
try:
    with urllib.request.urlopen(geojson_url) as r:
        geojson_depts = json.load(r)
    print("✅ GeoJSON chargé")
except Exception as e:
    print(f"⚠️  Pas de connexion internet : {e}")
    print("   Téléchargez le fichier manuellement et chargez-le avec :")
    print("   with open('departements.geojson') as f: geojson_depts = json.load(f)")
    geojson_depts = None

✅ GeoJSON chargé


In [ ]:
if geojson_depts:
    df_dept_moy = df.groupby("dept")[list(Y_COLS.values())].mean().reset_index()

    for patho, col in Y_COLS.items():
        fig = px.choropleth(
            df_dept_moy,
            geojson=geojson_depts,
            locations="dept",
            featureidkey="properties.code",
            color=col,
            color_continuous_scale="Reds",
            title=f"Taux moyen urgences {patho} par département (2020-2025)",
            labels={col: "Taux /100k hab."},
            hover_data={"dept": True, col: ":.1f"}
        )
        fig.update_geos(fitbounds="locations", visible=False)
        fig.update_layout(
            margin={"r": 0, "t": 40, "l": 0, "b": 0},
            height=550,
            template="plotly_white"
        )
        fig.show()
        fig.write_html(EDA_DIR / f"eda_04_carte_{patho.lower()}.html")
        print(f"💾 eda/eda_04_carte_{patho.lower()}.html")

💾 eda_04_carte_allergie.html


💾 eda_04_carte_asthme.html


💾 eda_04_carte_bronchiolite.html


### 2.2 Classement des départements — graphique interactif

In [ ]:
df_dept_moy = df.groupby("dept")[list(Y_COLS.values())].mean().reset_index()

for patho, col in Y_COLS.items():
    df_sorted = df_dept_moy.sort_values(col, ascending=True)

    fig = go.Figure(go.Bar(
        x=df_sorted[col],
        y=df_sorted["dept"],
        orientation="h",
        marker=dict(
            color=df_sorted[col],
            colorscale="Reds",
            showscale=True,
            colorbar=dict(title="Taux /100k")
        ),
        hovertemplate="Dept %{y}<br>Taux : %{x:.1f} /100k<extra></extra>"
    ))
    fig.update_layout(
        title=f"Classement des départements — Urgences {patho}",
        xaxis_title="Taux moyen /100k hab.",
        yaxis_title="Département",
        height=900,
        template="plotly_white"
    )
    fig.show()
    fig.write_html(EDA_DIR / f"eda_05_classement_{patho.lower()}.html")
    print(f"💾 eda/eda_05_classement_{patho.lower()}.html")

💾 eda_05_classement_allergie.html


💾 eda_05_classement_asthme.html


💾 eda_05_classement_bronchiolite.html


---
## 3. 🔗 Analyse de corrélation
### 3.1 Matrice de corrélation interactive

In [ ]:
FEATURES_CORR = [
    "taux_urgences_allergie", "taux_urgences_asthme", "taux_urgences_bronchiolite",
    "temp_moy", "temp_min", "temp_max", "humidite_moy", "vent_moy", "precip_moy",
    "pm10_moy", "pm25_moy", "no2_moy", "o3_moy",
    "pollen_graminees_moy", "pollen_betula_moy",
    "pollen_ambrosia_moy", "pollen_artemisia_moy",
    "part_seniors", "part_jeunes", "tx_urbain", "densite_med_gen",
    "tx_cadres", "tx_ouvriers",
    "sin_mois", "cos_mois",
]

cols_dispo = [c for c in FEATURES_CORR if c in df.columns]
corr_matrix = df[cols_dispo].corr(method="pearson").round(3)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale="RdBu_r",
    zmid=0,
    zmin=-1, zmax=1,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 8},
    hovertemplate="%{y} × %{x}<br>r = %{z:.3f}<extra></extra>"
))
fig.update_layout(
    title="Matrice de corrélation de Pearson",
    height=750,
    width=850,
    template="plotly_white",
    xaxis=dict(tickangle=45)
)
fig.show()
fig.write_html(EDA_DIR / "eda_06_correlation_matrix.html")
print(f"💾 eda/eda_06_correlation_matrix.html")

💾 eda_06_correlation_matrix.html


### 3.2 Top corrélations avec les variables cibles

In [ ]:
# Bar chart des corrélations absolues pour chaque pathologie
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"Top corrélations — {p}" for p in Y_COLS]
)

for i, (patho, col) in enumerate(Y_COLS.items(), 1):
    if col not in corr_matrix.columns:
        continue

    top_corr = (
        corr_matrix[col]
        .drop(list(Y_COLS.values()), errors="ignore")
        .sort_values(key=abs, ascending=True)
        .tail(12)
    )

    bar_colors = [
        "#E74C3C" if v > 0 else "#2980B9"
        for v in top_corr.values
    ]

    fig.add_trace(
        go.Bar(
            y=top_corr.index.tolist(),
            x=top_corr.values,
            orientation="h",
            marker_color=bar_colors,
            name=patho,
            hovertemplate="%{y}<br>r = %{x:.3f}<extra></extra>"
        ),
        row=1, col=i
    )

fig.update_layout(
    title_text="Top 12 corrélations avec chaque variable cible (rouge=positif, bleu=négatif)",
    height=550,
    showlegend=False,
    template="plotly_white"
)
fig.update_xaxes(range=[-1, 1], title_text="Corrélation de Pearson")
fig.show()
fig.write_html(EDA_DIR / "eda_07_top_correlations.html")
print(f"💾 eda/eda_07_top_correlations.html")

💾 eda_07_top_correlations.html


### 3.3 Scatter interactif — météo × urgences

In [ ]:
if "temp_moy" in df.columns:
    # Scatter : température vs taux d'urgences, coloré par mois
    df_scatter = df[["temp_moy", "mois", "dept"] + list(Y_COLS.values())].dropna()
    df_scatter["mois_label"] = df_scatter["mois"].apply(lambda m: MOIS_LABELS[m-1])

    for patho, col in Y_COLS.items():
        fig = px.scatter(
            df_scatter,
            x="temp_moy",
            y=col,
            color="mois_label",
            hover_data=["dept", "mois_label"],
            trendline="ols",
            title=f"Température moyenne vs Urgences {patho}",
            labels={
                "temp_moy": "Température moyenne (°C)",
                col: "Taux /100k hab.",
                "mois_label": "Mois"
            },
            opacity=0.5,
            template="plotly_white",
            height=500
        )
        fig.show()
        fig.write_html(EDA_DIR / f"eda_08_scatter_temp_{patho.lower()}.html")
    print("💾 Scatter température sauvegardés")

💾 Scatter température sauvegardés


### 3.4 Pollen × allergie — double axe interactif

In [ ]:
pollen_cols = [c for c in ["pollen_graminees_moy", "pollen_betula_moy",
                            "pollen_ambrosia_moy"]
               if c in df.columns]

if pollen_cols and "taux_urgences_allergie" in df.columns:
    df_pol = df.groupby("mois")[
        pollen_cols + ["taux_urgences_allergie"]
    ].mean().reset_index()
    df_pol["mois_label"] = df_pol["mois"].apply(lambda m: MOIS_LABELS[m-1])

    POLLEN_COLORS = {
        "pollen_graminees_moy": "#27AE60",
        "pollen_betula_moy":    "#8E44AD",
        "pollen_ambrosia_moy":  "#E67E22",
    }
    POLLEN_NAMES = {
        "pollen_graminees_moy": "Graminées",
        "pollen_betula_moy":    "Bouleau",
        "pollen_ambrosia_moy":  "Ambroisie",
    }

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Barres pollen
    for col in pollen_cols:
        fig.add_trace(
            go.Bar(
                x=df_pol["mois_label"],
                y=df_pol[col],
                name=POLLEN_NAMES[col],
                marker_color=POLLEN_COLORS[col],
                opacity=0.6,
                hovertemplate="%{x}<br>%{y:.1f} grains/m³<extra></extra>"
            ),
            secondary_y=False
        )

    # Courbe urgences (axe droit)
    fig.add_trace(
        go.Scatter(
            x=df_pol["mois_label"],
            y=df_pol["taux_urgences_allergie"],
            name="Urgences Allergie",
            line=dict(color="#E74C3C", width=3),
            mode="lines+markers",
            marker=dict(size=8),
            hovertemplate="%{x}<br>Taux : %{y:.1f} /100k<extra></extra>"
        ),
        secondary_y=True
    )

    fig.update_layout(
        title_text="Saisonnalité pollen vs urgences allergie (moyenne nationale)",
        barmode="stack",
        template="plotly_white",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.update_yaxes(title_text="Concentration pollen (grains/m³)", secondary_y=False)
    fig.update_yaxes(title_text="Taux urgences allergie /100k hab.", secondary_y=True)

    fig.show()
    fig.write_html(EDA_DIR / "eda_09_pollen_vs_allergie.html")
    print(f"💾 eda/eda_09_pollen_vs_allergie.html")

💾 eda_09_pollen_vs_allergie.html


### 3.5 Qualité de l'air × asthme

In [ ]:
air_cols = [c for c in ["pm10_moy", "no2_moy", "o3_moy"]
            if c in df.columns]

if air_cols and "taux_urgences_asthme" in df.columns:
    df_air = df.groupby("mois")[
        air_cols + ["taux_urgences_asthme"]
    ].mean().reset_index()
    df_air["mois_label"] = df_air["mois"].apply(lambda m: MOIS_LABELS[m-1])

    AIR_COLORS = {
        "pm10_moy": "#7F8C8D",
        "no2_moy":  "#E67E22",
        "o3_moy":   "#3498DB",
    }
    AIR_NAMES = {
        "pm10_moy": "PM10 (µg/m³)",
        "no2_moy":  "NO₂ (µg/m³)",
        "o3_moy":   "O₃ (µg/m³)",
    }

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for col in air_cols:
        fig.add_trace(
            go.Scatter(
                x=df_air["mois_label"],
                y=df_air[col],
                name=AIR_NAMES[col],
                line=dict(color=AIR_COLORS[col], width=2, dash="dash"),
                mode="lines+markers",
                hovertemplate="%{x}<br>%{y:.1f} µg/m³<extra></extra>"
            ),
            secondary_y=False
        )

    fig.add_trace(
        go.Scatter(
            x=df_air["mois_label"],
            y=df_air["taux_urgences_asthme"],
            name="Urgences Asthme",
            line=dict(color="#2980B9", width=3),
            mode="lines+markers",
            marker=dict(size=8),
            hovertemplate="%{x}<br>Taux : %{y:.1f} /100k<extra></extra>"
        ),
        secondary_y=True
    )

    fig.update_layout(
        title_text="Qualité de l'air vs urgences asthme (moyenne nationale)",
        template="plotly_white",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.update_yaxes(title_text="Concentration polluant (µg/m³)", secondary_y=False)
    fig.update_yaxes(title_text="Taux urgences asthme /100k hab.", secondary_y=True)

    fig.show()
    fig.write_html(EDA_DIR / "eda_10_air_vs_asthme.html")
    print(f"💾 eda/eda_10_air_vs_asthme.html")

💾 eda_10_air_vs_asthme.html


---
## 4. 🔧 Traitement des valeurs manquantes
### 4.1 Diagnostic interactif

In [ ]:
missing = (
    (df.isnull().mean() * 100)
    .sort_values(ascending=True)
)
missing = missing[missing > 0].sort_values(ascending=True)

bar_colors = [
    "#E74C3C" if v > 50 else "#F39C12" if v > 20 else "#27AE60"
    for v in missing.values
]

fig = go.Figure(go.Bar(
    x=missing.values,
    y=missing.index,
    orientation="h",
    marker_color=bar_colors,
    hovertemplate="%{y}<br>%{x:.1f}% manquant<extra></extra>"
))
fig.add_vline(x=20, line_dash="dash", line_color="#F39C12",
              annotation_text="20%", annotation_position="top")
fig.add_vline(x=50, line_dash="dash", line_color="#E74C3C",
              annotation_text="50%", annotation_position="top")
fig.update_layout(
    title="Taux de valeurs manquantes (vert < 20% | orange 20-50% | rouge > 50%)",
    xaxis_title="% valeurs manquantes",
    template="plotly_white",
    height=max(400, len(missing) * 22)
)
fig.show()
fig.write_html(EDA_DIR / "eda_11_missing_values.html")
print(f"💾 eda/eda_11_missing_values.html")

### 4.2 Imputation et sauvegarde du dataset nettoyé

In [ ]:
df_clean = df.copy()

# ── Météo : médiane mensuelle nationale ──────────────────────────────────────
# Raison : les depts sans station SYNOP reçoivent la valeur typique du mois
for col in ["temp_moy", "temp_min", "temp_max",
            "humidite_moy", "vent_moy", "precip_moy"]:
    if col in df_clean.columns:
        mediane = df_clean.groupby("mois")[col].transform("median")
        df_clean[col] = df_clean[col].fillna(mediane)

# ── Qualité de l'air : médiane mensuelle nationale ────────────────────────────
for col in ["pm10_moy", "pm25_moy", "no2_moy", "o3_moy",
            "no_moy", "so2_moy", "nb_jours_pm10_eleve"]:
    if col in df_clean.columns:
        mediane = df_clean.groupby("mois")[col].transform("median")
        df_clean[col] = df_clean[col].fillna(mediane)

# ── Pollen : médiane mensuelle + 0 hors saison ────────────────────────────────
for col in [c for c in df_clean.columns if c.startswith("pollen_")]:
    mediane = df_clean.groupby("mois")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(mediane).fillna(0)

# ── Vérification finale ───────────────────────────────────────────────────────
missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0]

print("Valeurs manquantes restantes :")
if missing_after.empty:
    print("  ✅ Aucune — dataset complet !")
else:
    for col, n in missing_after.items():
        pct = n / len(df_clean) * 100
        print(f"  ⚠️  {col:<40} {n:>5} lignes ({pct:.1f}%)")

# Sauvegarde
df_clean.to_parquet(TABLES_DIR / "df_model_clean.parquet", index=False)
print(f"\n✅ Sauvegardé → data/tables/df_model_clean.parquet")
print(f"   {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")
print("\n🚀 Prêt pour le clustering (03_clustering.ipynb)")

Valeurs manquantes restantes :
  ⚠️  taux_hosp_allergie                           5 lignes (0.1%)
  ⚠️  taux_sos_allergie                         3699 lignes (53.5%)
  ⚠️  taux_hosp_asthme                             5 lignes (0.1%)
  ⚠️  taux_sos_asthme                           3699 lignes (53.5%)
  ⚠️  taux_urgences_bronchiolite                   3 lignes (0.0%)
  ⚠️  taux_hosp_bronchiolite                     238 lignes (3.4%)
  ⚠️  taux_sos_bronchiolite                     3702 lignes (53.6%)
  ⚠️  nb_jours_eleve                            5226 lignes (75.6%)

✅ Sauvegardé → data/tables/df_model_clean.parquet
   6,912 lignes × 59 colonnes

🚀 Prêt pour le clustering (03_clustering.ipynb)


---
## 📁 Fichiers produits

| Fichier (dans `eda/`) | Description |
|---|---|
| `eda/eda_01_evolution_nationale.html` | Évolution temporelle des 3 pathologies |
| `eda/eda_02_saisonnalite.html` | Profil saisonnier moyen |
| `eda/eda_03_top5_depts_{patho}.html` | Top 5 depts par pathologie |
| `eda/eda_04_carte_{patho}.html` | Carte choroplèthe |
| `eda/eda_05_classement_{patho}.html` | Classement de tous les depts |
| `eda/eda_06_correlation_matrix.html` | Heatmap de corrélation |
| `eda/eda_07_top_correlations.html` | Top 12 corrélations par pathologie |
| `eda/eda_08_scatter_temp_{patho}.html` | Scatter température × urgences |
| `eda/eda_09_pollen_vs_allergie.html` | Pollen vs allergie (double axe) |
| `eda/eda_10_air_vs_asthme.html` | Qualité de l'air vs asthme |
| `eda/eda_11_missing_values.html` | Diagnostic valeurs manquantes |

| Fichier Parquet | Description |
|---|---|
| `data/tables/df_model_clean.parquet` | **Dataset nettoyé, prêt pour la modélisation** |

### Prochain notebook
→ `03_clustering.ipynb` — K-Means sur les variables environnementales